PRÁCTICA PARA CASA: Partiendo del fichero Registro Hoteles.xlsx (github). Realizaremos las transformaciones necesarias para resolver los siguientes enunciados:
- Ver en un gráfico circular los 3 Hoteles que más días de estancia han tenido (en %) y además el % del resto de Hoteles (será de todos los años)
- Ver en un gráfico de líneas el Importe Facturado (día pernoctado * precio base) de cada año y mes, por orden cronológico.
- Ver en un DF que guardaremos en un excel, los días de estancia por tipo de alojamiento y año.
- Ver la suma de Importe Facturado por Hotel después de aplicar un descuento del 15% si la fecha de pernoctación es temporada baja (Temporada baja = 01.10.año - 31/03.año+1). Hay que montar una fecha nueva (pista)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ruta_fichero = r"C:\PCAD\Registro Hoteles.xlsx"

# Creamos DF
registro = pd.read_excel(ruta_fichero, sheet_name='Entrada Huespedes')

# Ver en un gráfico circular los 3 Hoteles que más días de estancia han tenido (en %) y además el % del resto de Hoteles (será de todos los años)

## Calcular los días de pernoctación 
#Crear la función para obtener los días de pernoctación
def generar_fechas(fila):
    return pd.date_range(fila['FECHA ENTRADA'],fila['FECHA SALIDA'] - pd.Timedelta(days=1))

# Llamar a la función para generar las fechas
registro['Fecha'] = registro.apply(generar_fechas, axis=1)

# Con explode la lista de cada día
registro = registro.explode('Fecha')

# Crear las dos columnas, 'Año' y 'Mes'
registro['Año'] = registro['Fecha'].dt.year
registro['Mes'] = registro['Fecha'].dt.month_name(locale='es_ES.UTF-8')

## Agrupar por Hotel y contar los días sin diferencias el año
hoteles_dia = registro.groupby('NOMBRE HOTEL').Fecha.count().reset_index()
hoteles_dia

# Sacar los 3 hoteles con más pernoctación
hoteles_dia_top3 = hoteles_dia.nlargest(3,'Fecha')

# Calcular el resto de hoteles y agruparlos en 1 solo valor
hoteles_dia_resto = hoteles_dia.nsmallest(hoteles_dia['Fecha'].count() - 3, 'Fecha')
hoteles_resto = hoteles_dia_resto['Fecha'].sum()

# Hacer un DF para el resto de hoteles y la suma de los días de pernoctación
hoteles_df_resto = pd.DataFrame([{'NOMBRE HOTEL':'Resto Hoteles', 'Fecha':hoteles_resto}])

# Concatenar los 2 DF
resultado = pd.concat([hoteles_dia_top3,hoteles_df_resto], ignore_index=True)
resultado

## Hacer el gráfico
# Para crear un gráfico necesitamos reservar espacio cuadrado en el kernel, figsize(ancho,alto)
plt.figure(figsize=(6,6))

# %1(entero),2(decimales que deseo)
plt.pie(resultado['Fecha'], labels=resultado['NOMBRE HOTEL'],autopct='%1.2f%%')

# Añadir el título
plt.title('Días de Pernoctación por Hotel', fontsize=15, pad=30, loc='left')

# Mostrar el gráfico
plt.show()

In [ ]:
#Ver en un gráfico de líneas el Importe Facturado (día pernoctado * precio base) de cada año y mes, 
# por orden cronológico. 

importe_años = registro.groupby(['Año','Mes'])['PRECIO BASE/DIA'].sum().reset_index()
importe_años

## Ordenar los meses de manera cronologica, será según una lista
orden_meses = ['Enero','Febrero','Marzo','Abril','Mayo','Junio','Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

## Usaremos el método categorical para ordenar de manera categórica
importe_años['Mes'] = pd.Categorical(importe_años['Mes'],categories=orden_meses, ordered=True)
importe_años = importe_años.sort_values(by=['Año','Mes'])

#Renombro la columna PRECIO BASE/DIA
importe_años.rename(columns={'PRECIO BASE/DIA': 'Importe Facturado'}, inplace=True)

# Dibujar el gráfico de líneas --> me conecta todas las lineas
importe_años['Mes_Num'] = importe_años['Mes'].cat.codes

# Creamos la figura ax
fig, ax=plt.subplots(figsize=(10,4))

# Haremos un bucle para ir dibujando las líneas en función del año
for año in importe_años['Año'].unique():
    datos_año = importe_años[importe_años['Año'] == año]
    ax.plot(datos_año['Mes_Num'], datos_año['Importe Facturado'],marker='o', label=str(año))

# Ordenar el eje de las x
ax.set_xticks(range(len(orden_meses)))
ax.set_xticklabels(orden_meses, rotation=45)

# Incorporar titulo leyenda y titulo gráfico
ax.legend(title='Año')
plt.title('Importe Facturado por mes y por año', fontsize=15, pad=30, loc='center')

# Mostrar el gráfico
plt.show()

In [ ]:
# Forma más directa, hago PIVOT + T.plot
tabla_pivot = importe_años.pivot(index='Año', columns='Mes', values='Importe Facturado').fillna(0)
tabla_pivot.T.plot(marker='o')

In [ ]:
# Ver en un DF que guardaremos en un excel, los días de estancia por tipo de alojamiento y año.
import pandas as pd
ruta_fichero = r"C:\PCAD\Registro Hoteles.xlsx"
# Creamos DF
registro1 = pd.read_excel(ruta_fichero, sheet_name='Entrada Huespedes')

# Calcular los días de estancia 
registro1['Dias Estancia'] = (registro1['FECHA SALIDA'] - registro1['FECHA ENTRADA']).dt.days
registro1['Dias Estancia'] = registro1['Dias Estancia'] + 1 #para que me incluya el día de entrada como día de estancia 

# Calculamos el año, usando la fecha de entrada 
registro1['Año'] = registro1['FECHA ENTRADA'].dt.year

# Realizar la agrupación por tipo de alojamiento y año, sumando los días de estancias
registro_tipo_año = registro1.groupby(['TIPO ALOJAMIENTO','Año'])['Dias Estancia'].sum().reset_index()

# Ahora guardamos el dataframe en un fichero xlsx
ruta_fichero = r"C:\PCAD\registro_tipo_año.xlsx"
registro_tipo_año.to_excel(ruta_fichero,sheet_name='Datos',header=True ,index=False)

print("Terminado con éxito")

registro_tipo_año

In [ ]:
# CORREGIR ESTE PUES NO DEBÍA HACERLO CON EL MISMO DATAFRAME DE LOS EJERCICIOS ANTERIORES, EL EXPLOTADO!!!!! 
# Y DEBÍA CALCULAR LOS DÍAS DE ESTANCIA 

# Ver en un DF que guardaremos en un excel, los días de estancia por tipo de alojamiento y año.
dia_alojamiento_año = registro.groupby(['TIPO ALOJAMIENTO','Año']).Fecha.count().reset_index()
dia_alojamiento_año.rename(columns={'Fecha': 'Dias de Estancia'}, inplace=True)

# Ahora guardamos el dataframe en un fichero xlsx
ruta_fichero = r"C:\PCAD\dia_alojamiento_año.xlsx"
dia_alojamiento_año.to_excel(ruta_fichero,sheet_name='Estancia',header=True ,index=False)

print("Terminado con éxito")
dia_alojamiento_año

In [ ]:
# - Ver la suma de Importe Facturado por Hotel después de aplicar un descuento del 15% si la fecha de pernoctación es temporada baja 
# (Temporada baja = 01.10.año - 31/03.año+1). Hay que montar una fecha nueva (pista)
import pandas as pd

# Crear un Dataframe con las fechas de temporada baja 
temp_baja = pd.DataFrame({
    'Fecha_Inicio':['2017-10-01','2018-10-01','2019-10-01','2020-10-01','2021-10-01','2022-10-01','2023-10-01'],
    'Fecha_Fin':['2018-03-31','2019-03-31','2020-03-31','2021-03-31','2022-03-31','2023-03-31','2024-03-31']
})

# Convertir los 2 campos que son str a datetime
temp_baja['Fecha_Inicio'] = pd.to_datetime(temp_baja['Fecha_Inicio'])
temp_baja['Fecha_Fin'] = pd.to_datetime(temp_baja['Fecha_Fin'])

def generar_fechas(fila):
    return pd.date_range(fila['Fecha_Inicio'],fila['Fecha_Fin'])

temp_baja['Fecha'] = temp_baja.apply(generar_fechas, axis=1)
temp_baja = temp_baja.explode('Fecha')  #para que se genere la lista
temp_baja

# Añadimos la columna del descuento (ejemplo: 20%)
temp_baja['Dto. Temporada Baja'] = 0.15

# Hago merge con la tabla registros, how=left 
registro_temp_baja = pd.merge(registro, temp_baja, on="Fecha", how='left')

# Calcular el Precio Dia T.Baja
registro_temp_baja['Precio T.Baja'] = registro_temp_baja['PRECIO BASE/DIA']*(1-registro_temp_baja['Dto. Temporada Baja'])

# Importe Facturado por Hotel 
importe_fra_hotel_baja = registro_temp_baja.groupby(['NOMBRE HOTEL'])['Precio T.Baja'].sum().reset_index()
importe_fra_hotel_baja

registro.head(10)
importe_fra_hotel = registro_temp_baja.groupby(['NOMBRE HOTEL'])['PRECIO BASE/DIA'].sum().reset_index()
importe_fra_hotel

print("El total Importe Factura con el descuento de temporada baja aplicado es",registro_temp_baja['Precio T.Baja'].sum())
print("El total Importe Facturado sin descuento es",registro_temp_baja['PRECIO BASE/DIA'].sum())

print(importe_fra_hotel_baja)



In [ ]:
# OTRA FORMA - GENERAR EL DF CON UNA FUNCIÓN (RESUELTO CON IA)
# - Ver la suma de Importe Facturado por Hotel después de aplicar un descuento del 15% si la fecha de pernoctación es temporada baja 
# (Temporada baja = 01.10.año - 31/03.año+1). Hay que montar una fecha nueva (pista)

# Creamos una lista para guardar los rangos de cada año
bloques_temporada = []

# Iteramos de 2017 a 2023      --> función la hice con IA
for año in range(2017, 2024):
    inicio = f"{año}-10-01"
    fin = f"{año+1}-03-31" # El año siguiente
    
    # Generamos el rango de días completo para este bloque
    rango = pd.date_range(start=inicio, end=fin)
    bloques_temporada.append(pd.DataFrame({'Fecha': rango}))

# Unimos todos los bloques en un solo DataFrame
temporada_baja = pd.concat(bloques_temporada, ignore_index=True)

# Añadimos la columna del descuento (ejemplo: 20%)
temporada_baja['Dto. Temporada Baja'] = 0.15

# Hago merge con la tabla registros, how=left 
registro_temporada_baja = pd.merge(registro, temporada_baja, on="Fecha", how='left')

# Calcular el Precio Dia T.Baja
registro_temporada_baja['Precio T.Baja'] = registro_temporada_baja['PRECIO BASE/DIA']*(1-registro_temporada_baja['Dto. Temporada Baja'])

# Importe Facturado por Hotel 
importe_fra_hotel_baja = registro_temporada_baja.groupby(['NOMBRE HOTEL'])['Precio T.Baja'].sum().reset_index()
importe_fra_hotel_baja

registro.head(10)
importe_fra_hotel = registro_temporada_baja.groupby(['NOMBRE HOTEL'])['PRECIO BASE/DIA'].sum().reset_index()
importe_fra_hotel

print("El total Importe Factura con el descuento de temporada baja aplicado es",registro_temporada_baja['Precio T.Baja'].sum())
print("El total Importe Facturado sin descuento es",registro_temporada_baja['PRECIO BASE/DIA'].sum())
print(importe_fra_hotel_baja)